# Qwen-Image-2.1 LoRA pilot and generation preview

Companion to `QWEN_LORA.md`. Reads the pilot scores and grids written by `qwen_lora_eval.py` and shows sample
strips from the full Step 4 generation (`qwen_lora_full_run.sh`). CPU only; nothing here trains or generates.

Decisions (Steps 1-2): text-to-image = per-class LoRA, checkpoint 1500, single prompt, seed-only variety (Stage A);
edit = untuned Qwen with the 10 varied prompts (Stage B), because every edit LoRA learned to ignore the reference.

In [ ]:
import os, glob, json
import pandas as pd
from PIL import Image
from IPython.display import display, Markdown

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
PILOT = "/workspace/synderm/qwen_lora/pilot"
GEN = "/workspace/synderm/qwen_lora/generations"   # mirror of /dev/shm/qwen_lora_generations, synced every 20 min
REALS = "/root/data/fitzpatrick17k/finalfitz17k"
GPT = "gpt-image-2.5-sunburst-medium-varied-seed20261028-n16"
COLS = ["folder", "n_images", "label_match", "top_wrong_label", "within_dist", "real_within_dist", "ref_dist",
        "nn_real_dist", "real_nn_dist", "near_copies", "ita_shift"]

## 1. Pilot scores

`label_match`: share of images a real-only 9-label DINOv2 linear probe calls the folder's label (probe 5-fold CV
accuracy 0.51, so relative between folders). `within_dist`: mean pairwise DINOv2 cosine distance among the 10 images
of one real (`real_within_dist` = the same for random sets of 10 reals). `ref_dist`: distance to the real the image is
filed under (for edit, its reference). `nn_real_dist`: median distance to the nearest training-pool image of the label
(`real_nn_dist` = among reals); `near_copies`: share under 0.05. `ita_shift`: median skin-tone (ITA) change vs the
reference, degrees.

In [ ]:
for label in ["psoriasis", "basal-cell-carcinoma", "lupus-erythematosus"]:
    m = pd.read_csv(f"{PILOT}/eval-{label}/metrics.csv")
    m["folder"] = m["folder"].str.replace(GPT, "GPT-Image-varied").str.replace(f"-{label}", "")
    display(Markdown(f"### {label.replace('-', ' ')}"))
    display(m[COLS].round(3))

## 2. Pilot grids (6 reals x 10 slots, real image on the left)

Psoriasis: GPT Image varied prompts, untuned Qwen t2i, Qwen t2i LoRA checkpoints, untuned Qwen edit (single and varied
prompts) and the edit LoRA at checkpoint 1000. Then the chosen settings on BCC and lupus.

In [ ]:
def show(path, title, width=1400):
    im = Image.open(path); im.thumbnail((width, 10000))
    display(Markdown(f"**{title}**")); display(im)

for name in [GPT, "t2i-psoriasis-untuned-single", "t2i-psoriasis-ckpt500-single", "t2i-psoriasis-ckpt1500-single",
             "t2i-psoriasis-ckpt2000-single", "edit-psoriasis-untuned-single", "edit-psoriasis-untuned-varied",
             "edit-psoriasis-ckpt1000-single"]:
    show(f"{PILOT}/eval-psoriasis/grid__{name}.jpg", name.replace(GPT, "GPT-Image-varied"))

In [ ]:
for label in ["basal-cell-carcinoma", "lupus-erythematosus"]:
    for name in [GPT, f"t2i-{label}-untuned-single", f"t2i-{label}-ckpt1500-single", f"edit-{label}-untuned-varied"]:
        show(f"{PILOT}/eval-{label}/grid__{name}.jpg", f"{label}: " + name.replace(GPT, "GPT-Image-varied"))

## 3. Full generation (Step 4), sample strips

For each class: 3 pool images that already have all 10 synthetic images in the mirror, shown as real | t2i slots 0-4 |
edit slots 0-4. Classes not yet reached by the t2i run show edit only. Counts are from the mirror on `/workspace`.

In [ ]:
pool = pd.read_csv(f"{GEN}/pool9.csv")
done = {m: {h[:-4] for h in os.listdir(f"{GEN}/{m}/09")} if os.path.isdir(f"{GEN}/{m}/09") else set() for m in ["t2i", "edit"]}
print({m: f"{len(v)} of {len(pool)} pool images complete" for m, v in done.items()})

tile = 160
for label, g in pool.groupby("label"):
    rows = []
    for h in g["md5hash"]:
        if h in done["edit"] or h in done["t2i"]:
            rows.append(h)
        if len(rows) == 3: break
    if not rows: continue
    strip = Image.new("RGB", (tile * 11, tile * len(rows)), "white")
    for i, h in enumerate(rows):
        strip.paste(Image.open(f"{REALS}/{h}.jpg").convert("RGB").resize((tile, tile)), (0, i * tile))
        for j, m in enumerate(["t2i", "edit"]):
            for k in range(5):
                p = f"{GEN}/{m}/{k:02d}/{h}.png"
                if os.path.exists(p):
                    strip.paste(Image.open(p).convert("RGB").resize((tile, tile)), ((1 + j * 5 + k) * tile, i * tile))
    display(Markdown(f"**{label}** (real | t2i x5 | edit x5): t2i {sum(h in done['t2i'] for h in g['md5hash'])}/{len(g)}, "
                     f"edit {sum(h in done['edit'] for h in g['md5hash'])}/{len(g)} complete"))
    display(strip)